# ⛓️ Chains and Runnable Composition in LangChain

### Tools and Techniques in Data Science — LangChain Module

| | |
|---|---|
| **Difficulty** | ⭐⭐ Intermediate |
| **Estimated Time** | 90–120 minutes |
| **Prerequisites** | Notebooks 01 & 02 |

---

**Welcome!** In this notebook, you'll learn how to connect LangChain components into powerful pipelines using **LCEL** (LangChain Expression Language). Chains are the backbone of every LangChain application.

> 💡 **Data Science Focus:** You'll build a *Data Science Concept Explainer* and an *ML Pipeline Recommender* — both real-world tools.

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. Understand what chains are and why they matter
2. Master LCEL's pipe operator (`|`) for composing pipelines
3. Use `RunnablePassthrough` and `RunnableLambda` for custom logic
4. Build sequential and parallel processing chains
5. Create a complete *Data Science Concept Explainer* pipeline
6. Build an *ML Pipeline Recommender* from dataset description to model suggestions
7. Know when to use chains vs agents

---

## ⚙️ Setup

In [ ]:
# Install packages if needed (uncomment)
# !pip install langchain langchain-openai langchain-ollama python-dotenv pydantic

In [ ]:
import os
from dotenv import load_dotenv

# LangChain core
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Model integrations
from langchain_openai import ChatOpenAI

# Pydantic for structured output
from pydantic import BaseModel, Field

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
print("OpenAI API key:", "found" if api_key else "NOT SET — Ollama examples still work")
print("All imports successful!")

---

## 1. What is a Chain?

A **chain** connects multiple LangChain components into a **pipeline** where data flows from one step to the next.

### The Simplest Chain

```mermaid
flowchart TD
    I["Input"] --> P["Prompt Template"]
    P --> M["Model"]
    M --> O["Output Parser"]
    O --> R["Result"]
```

In code, this is:

```python
chain = prompt | model | parser
result = chain.invoke("Decision Trees")
```

The `|` operator connects components left to right, like a Unix pipe.

In [ ]:
# The most basic chain: prompt → model → parser
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a data science expert. Be concise."),
    ("human", "What is {topic}?")
])

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# The chain: prompt | model | parser
basic_chain = prompt | model | StrOutputParser()

# Invoke with input
result = basic_chain.invoke({"topic": "Decision Trees"})
print("Result:", result[:200])

### Why Chains Are Useful

| Benefit | Description |
|---|---|
| **Modularity** | Swap any component without changing others |
| **Reusability** | Build once, invoke many times with different inputs |
| **Composability** | Combine chains into larger chains |
| **Parallelism** | Run independent steps simultaneously |
| **Testability** | Test each component in isolation |
| **Streaming** | Stream intermediate results as they're generated |

> 💡 **Think of it like a factory assembly line:** Each station does one thing well, and the product moves to the next station.

---

## 2. Runnable Concepts

In LangChain, everything that can be invoked is a **Runnable**. The key Runnables are:

| Runnable | Purpose | When to Use |
|---|---|---|
| `ChatPromptTemplate` | Formats messages with variables | Every chain starts here |
| `ChatOpenAI` / `ChatOllama` | Calls the LLM | Always needed |
| `StrOutputParser` | Extracts text from AIMessage | For plain text output |
| `RunnablePassthrough` | Passes input unchanged | When you need to forward data |
| `RunnableLambda` | Wraps a Python function | For custom processing logic |
| `RunnableParallel` | Runs multiple runnables simultaneously | For parallel branches |

### The Pipe Operator (`|`)

```mermaid
flowchart LR
    A["Runnable A"] -->|"output becomes input"| B["Runnable B"]
    B -->|"output becomes input"| C["Runnable C"]
```

The `|` operator creates a **RunnableSequence** — each runnable's output feeds into the next.

In [ ]:
# Every LangChain component is a Runnable
print("ChatPromptTemplate is Runnable:", hasattr(prompt, 'invoke'))
print("ChatOpenAI is Runnable:", hasattr(model, 'invoke'))
print("StrOutputParser is Runnable:", hasattr(StrOutputParser(), 'invoke'))
print("RunnablePassthrough is Runnable:", hasattr(RunnablePassthrough(), 'invoke'))
print("RunnableLambda is Runnable:", hasattr(RunnableLambda(lambda x: x), 'invoke'))

---

## 3. RunnablePassthrough

`RunnablePassthrough` passes the input **unchanged** to the next step.

### Why Use It?

Sometimes you need to **forward** the original input to multiple branches while also processing it:

```mermaid
flowchart TD
    I["Input"] --> RP["RunnablePassthrough"]
    RP --> A["Branch A: Process"]
    RP --> B["Branch B: Original Input"]
    A --> C["Combined Result"]
    B --> C
```

In [ ]:
# RunnablePassthrough forwards input unchanged
passthrough = RunnablePassthrough()

# It returns whatever you pass in
result = passthrough.invoke("Hello, World!")
print("Result:", result)

# Useful for forwarding context in chains
chain_with_passthrough = (
    RunnablePassthrough()
    | ChatPromptTemplate.from_messages([
        ("system", "You are a data science expert."),
        ("human", "Explain {input} in one sentence.")
    ])
    | model
    | StrOutputParser()
)

result = chain_with_passthrough.invoke("random forest")
print("\nResult:", result)

---

## 4. RunnableLambda

`RunnableLambda` wraps any Python function into a Runnable that can be used in a chain.

### Why Use It?

You often need to **transform data** between steps — format strings, extract fields, do calculations.

```mermaid
flowchart LR
    A["Runnable A"] -->|"output"| RL["RunnableLambda\n(function)"]
    RL -->|"transformed output"| B["Runnable B"]
```

In [ ]:
# Wrap a Python function as a Runnable
def format_topic(input_text: str) -> str:
    """Transform the input before sending to the model."""
    return f"Please explain '{input_text}' in exactly 3 bullet points for a data science student."

format_chain = RunnableLambda(format_topic)
result = format_chain.invoke("gradient descent")
print("Formatted:", result)

In [ ]:
# RunnableLambda in a full chain
def add_context(input_dict: dict) -> dict:
    """Add extra context to the prompt."""
    return {
        "topic": input_dict["topic"],
        "level": input_dict.get("level", "intermediate"),
        "context": f"This is for a university-level data science course."
    }

enhanced_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a data science educator at {level} level."),
    ("human", "{context}\nExplain {topic} clearly.")
])

chain = (
    RunnableLambda(add_context)  # Add context
    | enhanced_prompt            # Format prompt
    | model                      # Call LLM
    | StrOutputParser()          # Extract text
)

result = chain.invoke({"topic": "XGBoost", "level": "beginner"})
print(result[:300])

---

## 5. Parallel Chains

`RunnableParallel` runs multiple branches **simultaneously** and returns a dict of results.

```mermaid
flowchart TD
    I["Input"] --> RP["RunnableParallel"]
    RP --> A["Branch A"]
    RP --> B["Branch B"]
    RP --> C["Branch C"]
    A --> D["Combined Dict"]
    B --> D
    C --> D
```

In [ ]:
# Parallel chains: generate multiple outputs simultaneously
definition_prompt = ChatPromptTemplate.from_template(
    "Give a one-sentence definition of {topic} for data scientists."
)

example_prompt = ChatPromptTemplate.from_template(
    "Give one real-world example of using {topic} in data science."
)

pros_prompt = ChatPromptTemplate.from_template(
    "List 2 advantages of using {topic} in one sentence each."
)

# Run all three branches in parallel
parallel_chain = RunnableParallel(
    definition=definition_prompt | model | StrOutputParser(),
    example=example_prompt | model | StrOutputParser(),
    advantages=pros_prompt | model | StrOutputParser()
)

result = parallel_chain.invoke({"topic": "Random Forest"})

print("DEFINITION:")
print(result["definition"])
print("\nEXAMPLE:")
print(result["example"])
print("\nADVANTAGES:")
print(result["advantages"])

### 🔍 What Happened?

All three branches ran **simultaneously** (not sequentially)! This is faster than running them one by one.

The result is a dict with keys matching the branch names:
```python
result = {
    "definition": "Random Forest is...",
    "example": "In healthcare, Random Forest...",
    "advantages": "1. Handles overfitting..."
}
```

---

## 6. Sequential Processing

Sequential chains process data **step by step**, where each step's output feeds into the next.

### Chain Chaining: Output of One Chain Feeds Into Another

In [ ]:
# Step 1: Generate a definition
definition_chain = (
    ChatPromptTemplate.from_template("Define {topic} for data scientists in one sentence.")
    | model
    | StrOutputParser()
)

# Step 2: Use the definition to generate quiz questions
quiz_chain = (
    ChatPromptTemplate.from_template(
        "Based on this definition: {definition}\n\nGenerate 3 multiple-choice questions about {topic}."
    )
    | model
    | StrOutputParser()
)

# Chain them together: definition → quiz
# RunnablePassthrough forwards the original 'topic' alongside the definition
full_chain = (
    {"definition": definition_chain, "topic": RunnablePassthrough()}
    | quiz_chain
)

result = full_chain.invoke("Decision Trees")
print(result[:500])

### 🔍 What Happened?

```
Input: "Decision Trees"
    ↓
definition_chain → "A Decision Tree is a flowchart-like model that..."
    ↓
quiz_chain → "Q1: ... Q2: ... Q3: ..."
```

The `{"definition": ..., "topic": RunnablePassthrough()}` pattern:
- Runs `definition_chain` to get the definition
- Forwards the original `topic` via `RunnablePassthrough`
- Both are passed as a dict to `quiz_chain`

---

## 7. Branching and Composition

You can combine sequential and parallel patterns to build complex pipelines.

### Pattern: Fan-Out → Process → Fan-In

In [ ]:
# Fan-out: generate content in parallel
# Fan-in: combine into a final result

definition_prompt = ChatPromptTemplate.from_template(
    "Define {topic} for a data science student."
)
example_prompt = ChatPromptTemplate.from_template(
    "Give a real-world example of {topic} in data science."
)
limitation_prompt = ChatPromptTemplate.from_template(
    "List one limitation of {topic}."
)

# Fan-out: parallel branches
parallel = RunnableParallel(
    definition=definition_prompt | model | StrOutputParser(),
    example=example_prompt | model | StrOutputParser(),
    limitation=limitation_prompt | model | StrOutputParser()
)

# Fan-in: combine results into a final summary
def combine_results(input_dict: dict) -> str:
    """Combine parallel results into a formatted summary."""
    return (
        f"Definition: {input_dict['definition']}\n\n"
        f"Example: {input_dict['example']}\n\n"
        f"Limitation: {input_dict['limitation']}"
    )

full_pipeline = parallel | RunnableLambda(combine_results)

result = full_pipeline.invoke({"topic": "K-Means Clustering"})
print(result)

---

## 8. Project: Data Science Concept Explainer

Let's build a complete pipeline that takes a concept name and generates a comprehensive explanation.

### Pipeline Design

```mermaid
flowchart TD
    T["Topic"] --> EC["Explanation Chain"]
    T --> XC["Example Chain"]
    T --> QC["Quiz Chain"]
    EC --> J["Join Results"]
    XC --> J
    QC --> J
    J --> F["Final Formatted Result"]
```

### Schema

In [ ]:
# Structured output schema
class ConceptExplanation(BaseModel):
    """A complete explanation of a data science concept."""

    definition: str = Field(description="Clear definition in 1-2 sentences")
    intuition: str = Field(description="Intuitive explanation or analogy")
    python_example: str = Field(description="A short Python code snippet demonstrating the concept")
    use_case: str = Field(description="A real-world use case")
    advantages: list[str] = Field(description="List of 2-3 advantages")
    limitations: list[str] = Field(description="List of 2-3 limitations")
    difficulty: str = Field(description="beginner, intermediate, or advanced")

print("Schema defined:", list(ConceptExplanation.model_fields.keys()))

In [ ]:
# Build the Concept Explainer pipeline

# Chain 1: Generate explanation with structured output
explanation_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a data science educator. Provide comprehensive explanations."),
    ("human", "Explain the concept: {topic}")
])

explanation_chain = (
    explanation_prompt
    | model.with_structured_output(ConceptExplanation)
)

# Chain 2: Generate quiz questions
quiz_prompt = ChatPromptTemplate.from_template(
    "Generate 3 multiple-choice quiz questions about {topic} with answers."
)
quiz_chain = quiz_prompt | model | StrOutputParser()

# Combine: parallel explanation + quiz, then format
parallel_explainer = RunnableParallel(
    explanation=explanation_chain,
    quiz=quiz_chain
)

# Format the final output
def format_concept_output(input_dict: dict) -> str:
    """Format the structured explanation into readable text."""
    exp = input_dict["explanation"]
    quiz = input_dict["quiz"]
    
    lines = [
        f"=== {exp.definition} ===",
        f"",
        f"INTUITION:",
        f"{exp.intuition}",
        f"",
        f"PYTHON EXAMPLE:",
        f"{exp.python_example}",
        f"",
        f"USE CASE:",
        f"{exp.use_case}",
        f"",
        f"ADVANTAGES:",
    ]
    for adv in exp.advantages:
        lines.append(f"  + {adv}")
    lines.append(f"")
    lines.append(f"LIMITATIONS:")
    for lim in exp.limitations:
        lines.append(f"  - {lim}")
    lines.append(f"")
    lines.append(f"DIFFICULTY: {exp.difficulty}")
    lines.append(f"")
    lines.append(f"QUIZ:")
    lines.append(quiz)
    
    return "\n".join(lines)

# Full pipeline
concept_explainer = parallel_explainer | RunnableLambda(format_concept_output)

# Run it!
result = concept_explainer.invoke("Decision Trees")
print(result)

### 🔍 What Happened?

The pipeline did **everything in parallel**:
1. Generated a structured explanation (definition, intuition, code, pros/cons)
2. Generated quiz questions
3. Combined everything into a formatted result

The structured output means you can also **access fields programmatically**:

In [ ]:
# You can also work with the raw structured data
raw_result = parallel_explainer.invoke("Random Forest")

exp = raw_result["explanation"]
print(f"Concept: {exp.definition}")
print(f"Difficulty: {exp.difficulty}")
print(f"Has {len(exp.advantages)} advantages")
print(f"Quiz length: {len(raw_result['quiz'])} chars")

---

## 9. Project: ML Pipeline Recommender

A pipeline that takes a **dataset description** and recommends a complete ML approach.

### Pipeline Design

```mermaid
flowchart TD
    D["Dataset Description"] --> P1["1. Identify ML Problem"]
    P1 --> P2["2. Suggest Preprocessing"]
    P1 --> P3["3. Suggest Models"]
    P1 --> P4["4. Suggest Metrics"]
    P2 --> C["Combine All Recommendations"]
    P3 --> C
    P4 --> C
    C --> R["Complete ML Plan"]
```

This pipeline chains **sequential** and **parallel** patterns.

In [ ]:
# ML Pipeline Recommender

# Step 1: Identify the ML problem type
problem_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an ML consultant. Analyze datasets and identify the ML problem type."),
    ("human", "Given this dataset description, identify the ML problem type and target variable:\n\n{description}")
])
problem_chain = problem_prompt | model | StrOutputParser()

# Step 2: Suggest preprocessing (takes problem analysis as input)
preprocessing_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an ML engineer specializing in data preprocessing."),
    ("human", "Based on this problem analysis:\n{problem_analysis}\n\nSuggest specific preprocessing steps.")
])
preprocessing_chain = preprocessing_prompt | model | StrOutputParser()

# Step 3: Suggest models (takes problem analysis as input)
models_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an ML consultant. Recommend specific models."),
    ("human", "Based on this problem analysis:\n{problem_analysis}\n\nSuggest 3 specific ML models with brief justification.")
])
models_chain = models_prompt | model | StrOutputParser()

# Step 4: Suggest evaluation metrics (takes problem analysis as input)
metrics_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an ML evaluation expert."),
    ("human", "Based on this problem analysis:\n{problem_analysis}\n\nSuggest the best evaluation metrics and explain why.")
])
metrics_chain = metrics_prompt | model | StrOutputParser()

# Build the full pipeline
# Step 1: Identify problem (sequential)
# Steps 2-4: Run in parallel (fan-out)
ml_recommender = (
    # Step 1: Sequential - identify the problem
    {"problem_analysis": problem_chain}
    | RunnableParallel(
        # Steps 2-4: Parallel - all depend on problem analysis
        preprocessing=preprocessing_prompt | model | StrOutputParser(),
        models=models_prompt | model | StrOutputParser(),
        metrics=metrics_prompt | model | StrOutputParser()
    )
)

# Run it!
result = ml_recommender.invoke({
    "description": "I have a dataset with 50,000 customer records. Each record has: age, income, \
    purchase_history, browsing_time, and a column 'churned' (0/1) indicating whether \
    the customer stopped using our service in the last 3 months."
})

print("=" * 60)
print("PROBLEM ANALYSIS:")
print(result["problem_analysis"][:200])
print("\n" + "=" * 60)
print("PREPROCESSING:")
print(result["preprocessing"][:200])
print("\n" + "=" * 60)
print("RECOMMENDED MODELS:")
print(result["models"][:200])
print("\n" + "=" * 60)
print("EVALUATION METRICS:")
print(result["metrics"][:200])

### 🔍 What Happened?

```
Dataset description
    ↓ (Step 1: Sequential)
Problem analysis: "Binary classification problem..."
    ↓ (Steps 2-4: Parallel)
    ├── Preprocessing: "Handle missing values, encode categories..."
    ├── Models: "1. XGBoost, 2. Random Forest, 3. Logistic Regression..."
    └── Metrics: "1. F1-score, 2. AUC-ROC, 3. Precision-Recall..."
```

Step 1 runs **first** (sequential), then Steps 2-4 run **in parallel** (fan-out).

---

## 10. Ollama Implementation

Same pipelines, running locally with Ollama!

In [ ]:
from langchain_ollama import ChatOllama

# Local model
local_model = ChatOllama(model="llama3.2", temperature=0)

# Local Concept Explainer (simpler version without structured output)
local_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a data science educator."),
    ("human", "Explain {topic} with: 1) Definition 2) Intuition 3) Use case 4) One limitation")
])

local_explainer = local_prompt | local_model | StrOutputParser()

try:
    result = local_explainer.invoke({"topic": "Gradient Boosting"})
    print("Local Ollama Result:")
    print(result[:500])
except Exception as e:
    print(f"Ollama not available: {e}")

In [ ]:
# Local parallel chain
local_parallel = RunnableParallel(
    definition=ChatPromptTemplate.from_template("Define {topic} in one sentence.") | local_model | StrOutputParser(),
    example=ChatPromptTemplate.from_template("Give one example of {topic} in data science.") | local_model | StrOutputParser(),
    pros=ChatPromptTemplate.from_template("List 2 advantages of {topic}.") | local_model | StrOutputParser()
)

try:
    result = local_parallel.invoke({"topic": "Neural Networks"})
    print("Definition:", result["definition"][:100])
    print("\nExample:", result["example"][:100])
    print("\nPros:", result["pros"][:100])
except Exception as e:
    print(f"Ollama not available: {e}")

---

## 11. Chains vs Agents

When should you use a **chain** vs an **agent**?

| | **Chain** | **Agent** |
|---|---|---|
| **Flow** | Fixed, predictable | Dynamic, decides at runtime |
| **Steps** | Known in advance | Determined by the AI |
| **Speed** | Faster (no reasoning overhead) | Slower (AI decides each step) |
| **Cost** | Fewer LLM calls | More LLM calls |
| **Reliability** | High (deterministic flow) | Variable (AI may make mistakes) |
| **Best for** | Known workflows, pipelines | Tool use, exploration, complex reasoning |

### Use Chains When:
- You know the exact steps in advance
- The pipeline is the same every time
- You need speed and predictability
- Example: "Analyze this dataset → Generate report"

### Use Agents When:
- The steps depend on the input
- The AI needs to decide what tools to use
- You need dynamic, multi-step reasoning
- Example: "Answer this question using whatever tools are available"

> 💡 **Rule of thumb:** Start with chains. Only use agents when you need dynamic decision-making.

```mermaid
flowchart LR
    subgraph Chain
        C1["Step 1"] --> C2["Step 2"] --> C3["Step 3"]
    end
    subgraph Agent
        A1["Reason"] --> A2{"Choose tool"}
        A2 -->|"Tool A"| A3["Result"]
        A2 -->|"Tool B"| A4["Result"]
    end
```

---

## ⚠️ Common Mistakes

| Mistake | Problem | Fix |
|---|---|---|
| **Forgetting RunnablePassthrough** | Can't forward original input alongside transformed data | Use `{"key": chain, "original": RunnablePassthrough()}` pattern |
| **Parallel when sequential needed** | Steps depend on each other but run simultaneously | Use sequential chaining for dependent steps |
| **Too many LLM calls** | Slow and expensive | Combine prompts where possible |
| **No output parser** | Getting AIMessage objects instead of strings | Always add `StrOutputParser()` or use structured output |
| **Using agents for simple chains** | Unnecessary overhead and cost | Use chains for fixed, known workflows |
| **Not testing components** | Hard to debug broken chains | Test each component individually first |

In [ ]:
# Example: Testing components individually before combining
print("Testing individual components:")
print("=" * 40)

# 1. Test the prompt
test_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a data scientist."),
    ("human", "What is {topic}?")
])
formatted = test_prompt.invoke({"topic": "XGBoost"})
print("Prompt works:", "XGBoost" in formatted.messages[1].content)

# 2. Test the model
test_response = model.invoke(formatted)
print("Model works:", len(test_response.content) > 0)

# 3. Test the parser
test_parsed = StrOutputParser().invoke(test_response)
print("Parser works:", isinstance(test_parsed, str))

# 4. Now combine into a chain
full_chain = test_prompt | model | StrOutputParser()
print("Chain works:", len(full_chain.invoke({"topic": "XGBoost"})) > 0)
print("\nAll components verified!")

---

## 🏋️ Exercises

Complete these exercises to solidify your understanding.

### Exercise 1: Build a Study Plan Generator

Create a chain that:
1. Takes a topic and duration
2. Generates a structured study plan (using Pydantic)
3. Generates quiz questions based on the plan
4. Combines both into a formatted output

**Schema:**
```python
class StudyPlan(BaseModel):
    topic: str
    duration: str
    steps: list[str]  # List of study steps
    resources: list[str]  # Recommended resources
    milestone: str  # What you'll know after
```

In [ ]:
# Exercise 1: Your code here!
#
# Steps:
# 1. Define the Pydantic schema
# 2. Create a prompt + structured model chain
# 3. Create a quiz chain
# 4. Combine with RunnableParallel
# 5. Format and print the result


### Exercise 2: Multi-Perspective Analyzer

Build a pipeline that takes a dataset description and analyzes it from **3 perspectives** in parallel:

1. **Statistician** — What statistical tests are relevant?
2. **ML Engineer** — What models would work?
3. **Data Engineer** — What data quality issues might exist?

Combine all three perspectives into a single report.

In [ ]:
# Exercise 2: Your code here!
#
# Hints:
# - Use RunnableParallel with 3 branches
# - Each branch has its own system message for the perspective
# - Use RunnablePassthrough to forward the dataset description
# - Combine results with a RunnableLambda


### Exercise 3: Chained Refinement

Build a **three-step refinement chain**:

1. **Draft** — Generate a rough explanation of a concept
2. **Critique** — Critique the draft for accuracy and clarity
3. **Final** — Rewrite based on the critique

Each step feeds into the next. This demonstrates true sequential chaining.

In [ ]:
# Exercise 3: Your code here!
#
# Chain structure:
# draft_chain: concept → draft explanation
# critique_chain: draft → critique
# final_chain: draft + critique → improved explanation
#
# Use dict chaining to pass both draft and critique to the final step


### 🌟 Challenge: End-to-End ML Recommender

Build a complete ML recommendation pipeline that:

1. Takes a **dataset description**
2. Identifies the **problem type** (classification, regression, clustering, etc.)
3. In parallel, recommends:
   - Preprocessing steps
   - 3 candidate models with justification
   - Evaluation metrics
   - A visualization suggestion
4. Combines everything into a **formatted ML project proposal**
5. Generates a **Python starter script** skeleton

**Bonus:** Use Pydantic for the intermediate schema!

In [ ]:
# 🌟 Challenge: Your code here!
#
# Suggested pipeline:
# ml_pipeline = (
#     {"problem": problem_chain}
#     | RunnableParallel(
#         preprocessing=...,
#         models=...,
#         metrics=...,
#         visualization=...
#     )
#     | RunnableLambda(format_proposal)
#     | proposal_to_code_chain  # Extra chain for code generation
# )


---

## 📝 Key Takeaways

| Concept | What It Is | Key Insight |
|---|---|---|
| **Chain** | Connected pipeline of Runnables | Data flows left to right through `|` |
| **RunnablePassthrough** | Forwards input unchanged | Essential for parallel branches that need the original input |
| **RunnableLambda** | Wraps a Python function | For any custom transformation logic |
| **RunnableParallel** | Runs branches simultaneously | Faster than sequential for independent steps |
| **Fan-out / Fan-in** | Parallel then combine | Pattern: `parallel | combiner` |

### The Core Patterns

```python
# Pattern 1: Simple chain
chain = prompt | model | StrOutputParser()

# Pattern 2: With custom logic
chain = RunnableLambda(transform) | prompt | model | StrOutputParser()

# Pattern 3: Parallel branches
chain = RunnableParallel(a=chain_a, b=chain_b) | RunnableLambda(combine)

# Pattern 4: Sequential with passthrough
chain = {"result": chain_a, "input": RunnablePassthrough()} | chain_b
```

### Chains vs Agents

| Use Chains | Use Agents |
|---|---|
| Fixed, known workflow | Dynamic, decision-making needed |
| Speed matters | Exploration matters |
| Predictable costs | Variable complexity |

---

## 🚀 Next Steps

| Notebook | Topic | What You'll Learn |
|---|---|---|
| **01** | Introduction | What is LangChain? |
| **02** | Models, Prompts & Messages | Prompt engineering + structured output |
| **03** | LCEL & Chains | You are here! |
| **04** | Embeddings & Vector Stores | Semantic search and embeddings |
| **05** | RAG Applications | Question answering over documents |
| **06** | Tools & Agents | AI that can take actions |
| **07** | Advanced Project | Build a complete data science assistant |

---

🎉 **Excellent!** You've mastered LCEL and chains.

Next up: **Embeddings & Vector Stores** — where you'll learn semantic search! 🚀